In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/oof_lgb_fe_v2.npy
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/oof_lgb_baseline.npy
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/submission_phase4_lgb_fe_v2.csv
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/submission_phase3_lgb_baseline.csv
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/oof_logreg.npy
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/test_fe.parquet
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/__results__.html
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/features_v2.json
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/train_fe.parquet
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/__notebook__.ipynb
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/__output__.json
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/custom.css
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/__results___files/__results___15_1.png

## CELL 1 — Load OOF / test predictions from Phase 5

In [2]:
# === CELL 1 — load Phase 5 artifacts ===
from __future__ import annotations
import os, sys, json, time, logging
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from scipy.optimize import minimize
from scipy.stats import rankdata


# Bulletproof helper to find the exact folder containing the competition data
def get_kaggle_data_dir() -> Path:
    base_input = Path("/kaggle/input")
    if base_input.exists():
        for file_path in base_input.rglob("train.csv"):
            return file_path.parent
    return Path(".")


@dataclass(frozen=True)
class Config:
    seed: int = 42
    n_splits: int = 5
    target: str = "PitNextLap"
    id_col: str = "id"
    data_dir: Path = field(default_factory=get_kaggle_data_dir)
    out_dir: Path  = field(default_factory=lambda: Path("/kaggle/working"))


CFG = Config()


def find_artifact(name):
    # Use rglob to recursively search ALL subfolders
    for p in Path("/kaggle/input").rglob(name):
        if p.is_file():
            return p
            
    # Fallback: working dir
    p = CFG.out_dir / name
    return p if p.exists() else None


# Phase 5 outputs
oof_lgb  = np.load(find_artifact("p5_oof_lgb.npy"))
oof_xgb  = np.load(find_artifact("p5_oof_xgb.npy"))
oof_cb   = np.load(find_artifact("p5_oof_cb.npy"))
test_lgb = np.load(find_artifact("p5_test_lgb.npy"))
test_xgb = np.load(find_artifact("p5_test_xgb.npy"))
test_cb  = np.load(find_artifact("p5_test_cb.npy"))

train = pd.read_csv(CFG.data_dir / "train.csv")
test  = pd.read_csv(CFG.data_dir / "test.csv")
y = train[CFG.target].astype(int).values

print("=== Individual OOF AUC (Phase 5 results) ===")
for name, oof in [("lgb", oof_lgb), ("xgb", oof_xgb), ("cb", oof_cb)]:
    print(f"  {name}: {roc_auc_score(y, oof):.5f}")

=== Individual OOF AUC (Phase 5 results) ===
  lgb: 0.94509
  xgb: 0.94555
  cb: 0.94922


## CELL 2 — Strategy 1: Simple average

In [3]:
# === CELL 2 — simple average ===
oof_avg  = (oof_lgb + oof_xgb + oof_cb) / 3
test_avg = (test_lgb + test_xgb + test_cb) / 3

print(f"Simple average OOF AUC: {roc_auc_score(y, oof_avg):.5f}")

Simple average OOF AUC: 0.94821


## CELL 3 — Strategy 2: Rank average

In [4]:
# === CELL 3 — rank average ===
# Convert each model's probabilities to ranks before averaging.
# Robust to differences in calibration between models. Since AUC is
# rank-based, this is the most theoretically appropriate average.
def to_rank_norm(p):
    return rankdata(p) / len(p)


oof_rank_lgb = to_rank_norm(oof_lgb)
oof_rank_xgb = to_rank_norm(oof_xgb)
oof_rank_cb  = to_rank_norm(oof_cb)
oof_rank_avg = (oof_rank_lgb + oof_rank_xgb + oof_rank_cb) / 3

# For test predictions, rank within the test set
test_rank_lgb = to_rank_norm(test_lgb)
test_rank_xgb = to_rank_norm(test_xgb)
test_rank_cb  = to_rank_norm(test_cb)
test_rank_avg = (test_rank_lgb + test_rank_xgb + test_rank_cb) / 3

print(f"Rank average OOF AUC: {roc_auc_score(y, oof_rank_avg):.5f}")

Rank average OOF AUC: 0.94815


## CELL 4 — Strategy 3: Weighted blend, weights optimized on OOF

In [5]:
# === CELL 4 — optimized weighted blend (probability space) ===
def neg_auc(weights, preds, y):
    w = np.clip(weights, 0, None)
    if w.sum() == 0:
        return 0.0
    w = w / w.sum()
    blend = sum(w_i * p for w_i, p in zip(w, preds))
    return -roc_auc_score(y, blend)


preds_oof = [oof_lgb, oof_xgb, oof_cb]
res = minimize(neg_auc, x0=[1/3]*3, args=(preds_oof, y),
               method="Nelder-Mead",
               options={"xatol": 1e-6, "fatol": 1e-7, "maxiter": 500})
w_opt = np.clip(res.x, 0, None)
w_opt = w_opt / w_opt.sum()
print(f"Optimal weights: lgb={w_opt[0]:.3f}, xgb={w_opt[1]:.3f}, cb={w_opt[2]:.3f}")

oof_wblend  = w_opt[0]*oof_lgb  + w_opt[1]*oof_xgb  + w_opt[2]*oof_cb
test_wblend = w_opt[0]*test_lgb + w_opt[1]*test_xgb + w_opt[2]*test_cb
print(f"Weighted blend OOF AUC: {roc_auc_score(y, oof_wblend):.5f}")

# Same optimization in rank space
res_rank = minimize(neg_auc, x0=[1/3]*3, args=([oof_rank_lgb, oof_rank_xgb, oof_rank_cb], y),
                    method="Nelder-Mead",
                    options={"xatol": 1e-6, "fatol": 1e-7, "maxiter": 500})
w_rank_opt = np.clip(res_rank.x, 0, None); w_rank_opt = w_rank_opt / w_rank_opt.sum()
print(f"Optimal rank-space weights: lgb={w_rank_opt[0]:.3f}, xgb={w_rank_opt[1]:.3f}, cb={w_rank_opt[2]:.3f}")
oof_wrank_blend  = (w_rank_opt[0]*oof_rank_lgb + w_rank_opt[1]*oof_rank_xgb + w_rank_opt[2]*oof_rank_cb)
test_wrank_blend = (w_rank_opt[0]*test_rank_lgb + w_rank_opt[1]*test_rank_xgb + w_rank_opt[2]*test_rank_cb)
print(f"Weighted RANK blend OOF AUC: {roc_auc_score(y, oof_wrank_blend):.5f}")

Optimal weights: lgb=0.065, xgb=0.127, cb=0.808
Weighted blend OOF AUC: 0.94943
Optimal rank-space weights: lgb=0.069, xgb=0.120, cb=0.811
Weighted RANK blend OOF AUC: 0.94940


## CELL 5 — Strategy 4: Stacked generalization

In [6]:
# === CELL 5 — stacking with logistic meta-learner ===
# Build a meta-model that uses the three OOF predictions as features.
# Train on a separate CV to avoid double-leakage.

skf = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed + 1)

X_meta_train = np.column_stack([oof_lgb, oof_xgb, oof_cb])
X_meta_test  = np.column_stack([test_lgb, test_xgb, test_cb])

oof_stack = np.zeros(len(y), dtype=np.float32)
test_stack = np.zeros(len(X_meta_test), dtype=np.float32)
for fold, (tr, va) in enumerate(skf.split(X_meta_train, y)):
    meta = LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs")
    meta.fit(X_meta_train[tr], y[tr])
    oof_stack[va] = meta.predict_proba(X_meta_train[va])[:, 1]
    test_stack += meta.predict_proba(X_meta_test)[:, 1] / CFG.n_splits
print(f"Stacked (logistic) OOF AUC: {roc_auc_score(y, oof_stack):.5f}")

# Coefficients tell us what the meta-learner picked
final_meta = LogisticRegression(C=1.0, max_iter=1000).fit(X_meta_train, y)
print(f"Meta-model coefficients (lgb, xgb, cb): {final_meta.coef_[0].round(3)}")
print(f"Meta-model intercept: {final_meta.intercept_[0]:.3f}")

Stacked (logistic) OOF AUC: 0.94941
Meta-model coefficients (lgb, xgb, cb): [0.685 0.909 5.206]
Meta-model intercept: -3.592


## CELL 6 — Honest comparison and final selection

In [7]:
# === CELL 6 — honest comparison ===
all_strategies = {
    "lgb_only":          (oof_lgb,         test_lgb),
    "xgb_only":          (oof_xgb,         test_xgb),
    "cb_only":           (oof_cb,          test_cb),
    "simple_avg":        (oof_avg,         test_avg),
    "rank_avg":          (oof_rank_avg,    test_rank_avg),
    "weighted_prob":     (oof_wblend,      test_wblend),
    "weighted_rank":     (oof_wrank_blend, test_wrank_blend),
    "stacked_logistic":  (oof_stack,       test_stack),
}

print(f"{'Strategy':<25} {'OOF AUC':>10}")
print("-"*40)
results = {}
for name, (oof, _) in all_strategies.items():
    s = roc_auc_score(y, oof)
    results[name] = s
    print(f"{name:<25} {s:>10.5f}")

best = max(results, key=results.get)
print(f"\n=> Best by OOF: {best}  ({results[best]:.5f})")

Strategy                     OOF AUC
----------------------------------------
lgb_only                     0.94509
xgb_only                     0.94555
cb_only                      0.94922
simple_avg                   0.94821
rank_avg                     0.94815
weighted_prob                0.94943
weighted_rank                0.94940
stacked_logistic             0.94941

=> Best by OOF: weighted_prob  (0.94943)


In [8]:
# === CELL 7 — submissions ===
# Submit two: best by OOF, and rank-average (safe choice).
# Kaggle lets you pick two final submissions; we'll use these.

def save_submission(test_probs, name):
    df = pd.DataFrame({CFG.id_col: test[CFG.id_col], CFG.target: test_probs})
    p = CFG.out_dir / f"submission_phase6_{name}.csv"
    df.to_csv(p, index=False)
    print(f"  Saved {p.name}  (head):")
    print(df.head(3).to_string(index=False))
    return p


_, best_test = all_strategies[best]
save_submission(best_test,       f"best_{best}")
save_submission(test_rank_avg,   "safe_rank_avg")
save_submission(test_wrank_blend, "weighted_rank")

  Saved submission_phase6_best_weighted_prob.csv  (head):
    id  PitNextLap
439140    0.005369
439141    0.005688
439142    0.004444
  Saved submission_phase6_safe_rank_avg.csv  (head):
    id  PitNextLap
439140    0.261497
439141    0.226610
439142    0.274568
  Saved submission_phase6_weighted_rank.csv  (head):
    id  PitNextLap
439140    0.303626
439141    0.301954
439142    0.283719


PosixPath('/kaggle/working/submission_phase6_weighted_rank.csv')